# 🚀 FaceFlux — 100% Free Full GPU Backend (Google Colab)
### NVIDIA T4 GPU (15GB VRAM) for Face Swap, Video Generation & Virtual Try-On

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Deependra1010verma/FaceFlux/blob/master/scripts/FaceFlux_Colab_GPU.ipynb)

---
### 📱 Mobile / Tablet se chalane ka aasan tarika:
1. Upar menu me check karein: **Runtime** -> **Change runtime type** -> **T4 GPU** select hona chahiye.
2. **Runtime** -> **Run all** (ya niche diye cells par Play ▶️) dabayein.
3. Cell 2 ke output me aakhri me ek **Clickable App Link** aayega, uspe tap karein — aapka Vercel app automatically open aur connect ho jayega!

In [ ]:
# [Cell 1] Setup environment, clone repository & download models
import os, sys

print("📥 [1/3] Cloning FaceFlux repository...")
if not os.path.exists("/content/FaceFlux"):
    !git clone https://github.com/Deependra1010verma/FaceFlux.git /content/FaceFlux
else:
    !cd /content/FaceFlux && git pull

%cd /content/FaceFlux

print("📦 [2/3] Installing dependencies & GPU support...")
!apt-get update -qq && apt-get install -y -qq ffmpeg aria2 > /dev/null 2>&1
!pip install -q -r backend/requirements.txt
!pip install -q onnxruntime-gpu diffusers transformers accelerate imageio[ffmpeg]

print("🤖 [3/3] Downloading AI models (InsightFace & CodeFormer)...")
!bash scripts/download-models.sh

# Install Cloudflare tunnel for reliable, zero-config public HTTPS URL
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✓ Setup complete! Ready to start GPU backend.")

In [ ]:
# [Cell 2] Start FaceFlux GPU Backend & Generate Direct Clickable Link
import os, time, subprocess, re

# Apne Vercel frontend ka link yaha daal sakte hain (optional):
VERCEL_APP_URL = "https://faceflux.vercel.app"

# Start FaceFlux backend with CUDA GPU enabled
print("🚀 Starting FaceFlux Backend on Colab GPU...")
backend_env = dict(os.environ, PORT="8000", HOST="0.0.0.0", GPU_PROVIDER="cuda")
backend_proc = subprocess.Popen(
    ["python3", "main.py"],
    cwd="/content/FaceFlux/backend",
    env=backend_env
)
time.sleep(4)

# Start Cloudflare Tunnel
print("🌐 Creating secure Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url = None
for _ in range(60):
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    direct_link = f"{VERCEL_APP_URL}/?server={tunnel_url}"
    print("\n" + "=" * 70)
    print("🎉 FaceFlux GPU Backend is ONLINE (15GB RAM + T4 GPU)!")
    print(f"📡 Direct Backend URL: {tunnel_url}")
    print("=" * 70)
    print(f"\n👉 PHONE / TABLET / BROWSER ME CHALANE KE LIYE IS LINK PAR CLICK KAREIN:")
    print(f"\n   {direct_link}\n")
    print("(Yeh link Vercel app ko kholega aur AUTOMATICALLY connect kar dega!)")
    print("=" * 70 + "\n")
    
    # Keep notebook alive
    while True:
        time.sleep(30)
else:
    print("Cloudflare tunnel connect nahi ho paya. localtunnel fallback try kar rahe hain:")
    !npx localtunnel --port 8000